# Step-Back Prompting
> Abstract a query to retrieve broader, principle-level context.

`StepBackTransformer` generates a more general version of the query
that retrieves foundational knowledge. The original and step-back
queries retrieve together, combining specific and general context.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.query import StepBackTransformer
from anchor.models import QueryBundle

## Define a Step-Back Function
The function takes a specific query and returns a more general version.
In production this would be an LLM call.

In [ ]:
def mock_step_back(query: str) -> str:
    """Generate a broader, more abstract version of the query."""
    return f"What are the general principles behind {query}?"

# Test the step-back function
original = "How does Anchor handle token budget overflow?"
stepped = mock_step_back(original)

print(f"Original:   {original}")
print(f"Step-back:  {stepped}")

## Create the Step-Back Transformer
Pass the generation function to `StepBackTransformer`.

In [ ]:
step_back = StepBackTransformer(generate_fn=mock_step_back)

print(f"Transformer: {type(step_back).__name__}")

## Transform a Query
`transform()` returns a list containing both the original and the
abstracted query.

In [ ]:
query = QueryBundle(query_str="What is context engineering?")
print(f"Original: {query.query_str}\n")

abstracted = step_back.transform(query)

print(f"Transformed to {len(abstracted)} queries:")
for i, q in enumerate(abstracted):
    print(f"  [{i}] {q.query_str}")

## Realistic Step-Back Examples
Step-back prompting works best when the original query is very specific
and would benefit from broader context.

In [ ]:
def realistic_step_back(query: str) -> str:
    """Simulate LLM-generated step-back queries."""
    step_backs = {
        "Why does my FAISS index return wrong results for cosine similarity?": (
            "How do vector similarity search algorithms work?"
        ),
        "How do I configure Anchor's sliding window memory with 4096 tokens?": (
            "What are the different memory management strategies for LLM context?"
        ),
        "Why is my RAG pipeline returning irrelevant chunks?": (
            "What factors affect retrieval quality in RAG systems?"
        ),
    }
    return step_backs.get(query, f"What are the general principles behind {query}?")

realistic = StepBackTransformer(generate_fn=realistic_step_back)

examples = [
    "Why does my FAISS index return wrong results for cosine similarity?",
    "How do I configure Anchor's sliding window memory with 4096 tokens?",
    "Why is my RAG pipeline returning irrelevant chunks?",
]

for q_str in examples:
    q = QueryBundle(query_str=q_str)
    results = realistic.transform(q)
    print(f"Specific:  {q_str}")
    print(f"Step-back: {results[-1].query_str}")
    print()

## Dual Retrieval Pattern
Use both the original and step-back queries for retrieval, then merge.

In [ ]:
def mock_retrieve(query_str: str) -> list:
    base = hash(query_str) % 100
    return [f"doc-{base + i}" for i in range(3)]

query = QueryBundle(query_str="Why is my RAG pipeline returning irrelevant chunks?")
results = realistic.transform(query)

print("Dual retrieval:\n")
all_docs = []
for q in results:
    docs = mock_retrieve(q.query_str)
    all_docs.extend(docs)
    label = "Original" if q == results[0] else "Step-back"
    print(f"  {label}: {q.query_str[:55]}")
    print(f"    -> {docs}")

unique = list(dict.fromkeys(all_docs))
print(f"\nTotal: {len(all_docs)} docs, {len(unique)} unique")
print(f"Unique docs: {unique}")

## Key Takeaways
- `StepBackTransformer` generates a broader version of specific queries
- Both original and step-back queries retrieve together
- Combines specific answers with foundational context
- Best for highly specific or troubleshooting queries
- The abstraction level depends on the generation function

**Next:** [Query Classifiers](05_classifiers.ipynb)